# __Methodology__
In this notebook, we try to replicate the methodology described in this paper: https://pmc.ncbi.nlm.nih.gov/articles/PMC9046851/. However, we are a little bit more lucky, as we can skip the first part in which they try to extract the patches that covers a biomarker of tumor, as we already have the masks. So, we will focus on the second part

## 🌐 **Google Drive Connection**

In [ ]:
from google.colab import drive
drive.mount("/gdrive", force_remount=True)
current_dir = "/gdrive/My\\ Drive/Project/Challenge\\ 2-v2"
%cd $current_dir

Mounted at /gdrive
/gdrive/My Drive/Project/Challenge 2-v2


## ⚙️ **Libraries Import**

In [ ]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader, Dataset

# Configurazione di TensorBoard e directory
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
import cv2
from tqdm import tqdm

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

#Lion optimizer
%pip install lion-pytorch

PyTorch version: 2.9.0+cpu
Device: cpu


## 🦠 **Data Cleaning**

In [ ]:
# Directory for dataset storage
DATA_DIR = "./train_data"
CSV_PATH = "./train_labels.csv"
CLEAN_DATA_DIR = "./clean_train_data"
CLEAN_CSV_DIR = "./clean_train_labels.csv"
PATCH_DIR = "./patch"
PATCH_CSV = "./patch.csv"
PATCH_TEST_DIR = "./test_patches_temp/images"

In [ ]:
def create_clean_dataset(original_df, source_dir, dest_dir, dest_csv, threshold=0.005):
    """
    Filters the dataset and copies valid images/masks to a new folder.
    """
    print(f"Creating clean dataset in: {dest_dir}")

    # Create the destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)

    clean_rows = []
    rejected_count = 0

    print(f"Scanning {len(original_df)} images...")

    for idx in tqdm(range(len(original_df))):
        try:
            # Get filenames
            img_name = original_df.iloc[idx, 0]
            mask_name = img_name.replace("img", "mask")

            src_img_path = os.path.join(source_dir, img_name)
            src_mask_path = os.path.join(source_dir, mask_name)

            # 1. READ IMAGE
            img = cv2.imread(src_img_path)
            if img is None: continue

            # 2. FILTER LOGIC (Shrek/Slime Detector)
            b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]

            # Define foreground (not white background)
            bg_mask = (b > 215) & (g > 215) & (r > 215)
            fg_count = np.sum(~bg_mask)

            is_clean = True

            # If image isn't empty, check for poison
            if fg_count > 0:
                bad_pixels = ((g > b + 20) | (g > r + 20)) & (~bg_mask)
                if (np.sum(bad_pixels) / fg_count) > threshold:
                    is_clean = False
                    rejected_count += 1

            # 3. SAVE IF CLEAN
            if is_clean:
                # Copy Image
                shutil.copy(src_img_path, os.path.join(dest_dir, img_name))

                # Copy Mask (Crucial! We need this for the Dataset class later)
                if os.path.exists(src_mask_path):
                    shutil.copy(src_mask_path, os.path.join(dest_dir, mask_name))

                # Keep the label record
                clean_rows.append(original_df.iloc[idx])

        except Exception as e:
            print(f"Error processing {img_name}: {e}")

    # Save the new CSV
    df_clean = pd.DataFrame(clean_rows)
    df_clean.to_csv(dest_csv, index=False)

    print(f"\nDone!")
    print(f"Rejected {rejected_count} poisoned images.")
    print(f"Saved {len(df_clean)} clean images/masks to '{dest_dir}'")

    return df_clean

In [ ]:
if os.path.exists(CLEAN_CSV_DIR) and os.path.exists(CLEAN_DATA_DIR):
    print(f"Found cached clean dataset at: {CLEAN_DATA_DIR}")
    print("Skipping scan and using existing files.")

    # Load the clean list
    df_clean = pd.read_csv(CLEAN_CSV_DIR)

else:
    print("Clean dataset not found. Creating it now...")
    original_df = pd.read_csv(CSV_PATH)

    # Run creation (Filter + Copy)
    df_clean = create_clean_dataset(original_df, DATA_DIR, CLEAN_DATA_DIR, CLEAN_CSV_DIR)


✅ Found cached clean dataset at: ./clean_train_data
   Skipping scan and using existing files.


## 🔎 **Data Loading**

In [ ]:
# 2. Define the dictionary to translate them
label_mapping = {
    "Luminal A": 0,
    "Luminal B": 1,
    "HER2(+)": 2,
    "Triple negative": 3
}

# 3. Apply the translation
df_clean['label'] = df_clean['label'].map(label_mapping)

# 4. Check for any mistakes (NaN means a spelling mismatch)
if df_clean['label'].isnull().any():
    print("WARNING: Some labels didn't match! Check spelling.")
    print(df_clean[df_clean['label'].isnull()])
else:
    print("Success! Labels converted to numbers:", df_clean['label'].unique())

Success! Labels converted to numbers: [3 1 0 2]


# __Patch extraction__

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from tqdm import tqdm

# MACENKO NORMALIZATION HELPER ---
# We define this helper function to be used inside the extraction loop.
# Standard Reference Mean/Std for H&E (Precomputed from Macenko paper)
MACENKO_TARGET_MEANS = np.array([1.9705, 1.6249])
MACENKO_TARGET_STDS  = np.array([0.2358, 0.1806])
MACENKO_REF_HE_MATRIX = np.array([
    [0.5626, 0.2159],
    [0.7293, 0.8012],
    [0.4062, 0.5581]
])

def normalize_stain(img_bgr):
    """
    Applies Macenko stain normalization to a single BGR image.
    Efficient implementation using precomputed targets.
    """
    if img_bgr is None: return None

    try:
        # Convert to RGB
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w, c = img.shape
        img = img.reshape((-1, 3))

        # Calculate Optical Density (OD)
        Io = 240
        beta = 0.15
        alpha = 1

        # Avoid division by zero
        OD = -np.log((img.astype(float) + 1) / Io)

        # Remove transparent pixels (glass)
        ODhat = OD[(OD > beta).all(axis=1)]

        # Safety: If image is mostly white/glass, return original
        if len(ODhat) < 10: return img_bgr

        # Compute Eigenvectors
        eigvals, eigvecs = np.linalg.eigh(np.cov(ODhat.T))

        # Project on plane spanned by 2 largest eigenvectors
        That = ODhat.dot(eigvecs[:, 1:3])

        # Find Min/Max Vectors
        phi = np.arctan2(That[:, 1], That[:, 0])
        minPhi = np.percentile(phi, alpha)
        maxPhi = np.percentile(phi, 100 - alpha)

        vMin = eigvecs[:, 1:3].dot(np.array([(np.cos(minPhi), np.sin(minPhi))]).T)
        vMax = eigvecs[:, 1:3].dot(np.array([(np.cos(maxPhi), np.sin(maxPhi))]).T)

        # Force vector 1 to be Hematoxylin (H) and 2 to be Eosin (E)
        if vMin[0] > vMax[0]:
            HE = np.array((vMin[:, 0], vMax[:, 0])).T
        else:
            HE = np.array((vMax[:, 0], vMin[:, 0])).T

        # Extract Stain Concentrations (C)
        Y = np.reshape(OD, (-1, 3)).T
        C = np.linalg.lstsq(HE, Y, rcond=None)[0]

        # Normalize Concentrations
        maxC = np.percentile(C, 99, axis=1)
        tmp = np.divide(maxC, MACENKO_TARGET_MEANS)
        C2 = np.divide(C, tmp[:, np.newaxis])

        # Reconstruct Image
        Inorm = Io * np.exp(-np.dot(MACENKO_REF_HE_MATRIX, C2))
        Inorm = np.reshape(Inorm.T, (h, w, 3)).clip(0, 255).astype(np.uint8)

        # Return in BGR
        return cv2.cvtColor(Inorm, cv2.COLOR_RGB2BGR)

    except Exception:
        return img_bgr # Fallback on error

In [ ]:
def araujo_contrast_stretching(img):
    """
    Performs histogram stretching as described in Araujo et al. 2017.
    It stretches the lower 90% of the data to fill the 0-255 range.
    Helps fix 'washed out' H&E slides.
    """
    img = img.astype(float)

    # Process RGB channels independently
    for i in range(3):
        channel = img[:, :, i]

        # Find 0th and 90th percentile
        p0 = np.min(channel)
        p90 = np.percentile(channel, 90)

        # Avoid division by zero
        if p90 > p0:
            # Stretch: anything < p0 becomes 0, anything > p90 becomes 255
            channel = (channel - p0) / (p90 - p0) * 255.0
            channel = np.clip(channel, 0, 255)

       img[:, :, i] = channel

     return img.astype(np.uint8)

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from tqdm import tqdm

def extract_high_fidelity_patches(df, source_dir, dest_dir, patch_size=224, stride=112, mask_ratio_threshold=0.01):
    """
    Extracts patches at NATIVE RESOLUTION.
    NO RESIZING is performed, preserving 100% of the pixel detail.

    Args:
        patch_size: 224 (Standard for ResNet) or 256.
        stride:     112 (50% overlap) to capture edge details.
    """
    img_dest = os.path.join(dest_dir, "images")
    mask_dest = os.path.join(dest_dir, "masks")
    os.makedirs(img_dest, exist_ok=True)
    os.makedirs(mask_dest, exist_ok=True)

    patch_data = []

    print(f"Slicing {len(df)} images (High Fidelity Mode)...")
    print(f"Resolution: Native (1:1)")
    print(f"Patch Size: {patch_size}x{patch_size}")

    class_counts = {}

    for idx in tqdm(range(len(df))):
        try:
            row = df.iloc[idx]
            img_name = row[0]
            label = row.get('label', row[1] if len(row) > 1 else 'Unknown')

            # Paths
            img_path = os.path.join(source_dir, img_name)
            mask_name = os.path.splitext(img_name)[0].replace("img", "mask") + ".png"
            mask_path = os.path.join(source_dir, mask_name)

            if not os.path.exists(mask_path):
                 mask_path = os.path.join(source_dir, img_name.replace("img", "mask"))

            img = cv2.imread(img_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if img is None or mask is None: continue

            # Decomment to apply Macenko+Araujo normalization
            #img = normalize_stain(img)
            #img = araujo_contrast_stretching(img)

            h, w, _ = img.shape

            patch_id = 0

            # --- LOOP OVER IMAGE ---
            for y in range(0, h - patch_size + 1, stride):
                for x in range(0, w - patch_size + 1, stride):

                    # 1. EXTRACT (Direct Crop)
                    img_patch = img[y:y+patch_size, x:x+patch_size]
                    mask_patch = mask[y:y+patch_size, x:x+patch_size]

                    # 2. FILTER (Strong Supervision)
                    # Only keep if mask shows tumor
                    tumor_pixels = np.count_nonzero(mask_patch)
                    tumor_ratio = tumor_pixels / (patch_size**2)

                    if tumor_ratio >= mask_ratio_threshold:

                        # 3. SAVE (NO RESIZING)
                        # We write the pixels exactly as they are on the slide.

                        p_img_name = f"{label}_{os.path.splitext(img_name)[0]}_p{patch_id}.png"
                        p_mask_name = f"{label}_{os.path.splitext(mask_name)[0]}_p{patch_id}.png"

                        cv2.imwrite(os.path.join(img_dest, p_img_name), img_patch)
                        cv2.imwrite(os.path.join(mask_dest, p_mask_name), mask_patch)

                        patch_data.append({
                            "filename": p_img_name,
                            "label": label,
                            "tumor_ratio": tumor_ratio,
                            "original_image": img_name
                        })

                        class_counts[label] = class_counts.get(label, 0) + 1

                    patch_id += 1

        except Exception as e:
            print(f"Error on {img_name}: {e}")

    new_df = pd.DataFrame(patch_data)
    new_df.to_csv(PATCH_CSV, index=False)

    print("\nExtraction Complete.")
    print("Class Distribution:", class_counts)
    return new_df

# __Train/Validation split and patch extraction__
We split the train and validation before the patch extraction. In this way we have consistency on the patches that will be in the two sets. Regarding the patches:
1. For every image in the list, the code normalizes it and then scans it with a sliding window ($224 \times 224$).

2. For every window, it asks: "Does the mask here show the problem (tumor)?"
  - Yes (>1%): It saves that specific patch (Image + Mask)
  - No: It discards it.

In this way we avoid the problem of having a 'healthy non-tumoral' patch, labelled, for example, as 'HER2(+)', as this may have an impact on the learning capabilities of the network.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Load your master list of slides
df = pd.read_csv(CLEAN_CSV_DIR)

# 2. Split at the SLIDE level
# This ensures that all patches from Slide X go to the same set.
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"Total Slides: {len(df)}")
print(f"Training Slides: {len(train_df)}")
print(f"Validation Slides: {len(val_df)}")

# Define your source and destination paths
SOURCE_DIR = CLEAN_DATA_DIR
BASE_DEST_DIR = PATCH_DIR

Total Slides: 577
Training Slides: 461
Validation Slides: 116


In [ ]:
# 3. Extract TRAINING patches (High Overlap)
# Stride 112 = 50% overlap. More data, better learning.
print("\n--- Processing TRAINING Set ---")
extract_high_fidelity_patches(
    df=train_df,
    source_dir=SOURCE_DIR,
    dest_dir=os.path.join(BASE_DEST_DIR, "train"),
    patch_size=256,
    stride=128,
    mask_ratio_threshold=0.01
)

# 4. Extract VALIDATION patches (No Overlap)
# Stride 224 = 0% overlap. Faster evaluation, honest metrics.
print("\n--- Processing VALIDATION Set ---")
extract_high_fidelity_patches(
    df=val_df,
    source_dir=SOURCE_DIR,
    dest_dir=os.path.join(BASE_DEST_DIR, "val"),
    patch_size=256,
    stride=256,
    mask_ratio_threshold=0.01
)


--- Processing TRAINING Set ---
🔬 Slicing 461 images (High Fidelity Mode)...
   Resolution: Native (1:1)
   Patch Size: 256x256


100%|██████████| 461/461 [07:28<00:00,  1.03it/s]



✅ Extraction Complete.
   Class Distribution: {'Triple negative': 905, 'Luminal A': 2184, 'Luminal B': 2834, 'HER2(+)': 2179}

--- Processing VALIDATION Set ---
🔬 Slicing 116 images (High Fidelity Mode)...
   Resolution: Native (1:1)
   Patch Size: 256x256


100%|██████████| 116/116 [01:09<00:00,  1.67it/s]


✅ Extraction Complete.
   Class Distribution: {'HER2(+)': 130, 'Luminal B': 193, 'Luminal A': 109, 'Triple negative': 50}


,filename,label,tumor_ratio,original_image
0,HER2(+)_img_0456_p4.png,HER2(+),0.048019,img_0456.png
1,HER2(+)_img_0456_p5.png,HER2(+),0.019745,img_0456.png
2,HER2(+)_img_0456_p13.png,HER2(+),0.095657,img_0456.png
3,HER2(+)_img_0456_p17.png,HER2(+),0.173370,img_0456.png
4,HER2(+)_img_0456_p18.png,HER2(+),0.027710,img_0456.png
...,...,...,...,...
477,Luminal A_img_0564_p5.png,Luminal A,0.022980,img_0564.png
478,Luminal A_img_0564_p6.png,Luminal A,0.097656,img_0564.png
479,Luminal A_img_0564_p8.png,Luminal A,0.046448,img_0564.png
480,Luminal A_img_0564_p9.png,Luminal A,0.067001,img_0564.png


## __Sanity check for patches extraction__
We can then run the following code, as a sanity check to see if at least one
patch is generated from each slide

In [ ]:
import os

# --- CONFIG ---
SOURCE_DIR = "./clean_train_data"
PATCH_DIR = "./patch"

PATCH_FOLDERS = [
    os.path.join(PATCH_DIR, "train/images"),
    os.path.join(PATCH_DIR, "val/images")
]

def debug_patches_coverage(source_dir, patch_folders):
    print("Starting DEBUG Coverage Check...")

    # 1. Get Source IDs
    if not os.path.exists(source_dir):
        print(f"Error: Source directory '{source_dir}' not found.")
        return

    source_files = [
        f for f in os.listdir(source_dir)
        if f.lower().endswith(('.tif', '.tiff', '.svs', '.jpg', '.png'))
        and "mask" not in f.lower()
    ]

    # Clean IDs
    source_ids = set(os.path.splitext(f)[0] for f in source_files)

    print(f"Found {len(source_ids)} source slides.")
    print(f"Examples (Source): {list(source_ids)[:5]}")

    # 2. Get Patch Filenames
    patch_files = []
    for p_folder in patch_folders:
        if os.path.exists(p_folder):
            files = [f for f in os.listdir(p_folder) if f.endswith('.png')]
            patch_files.extend(files)

    print(f"Found {len(patch_files)} patches total.")
    print(f"Examples (Patches): {patch_files[:5]}")

    # 3. Try Simple Matching
    found_patch_origins = set()

    print("\n--- Testing Matches ---")

    # We loop through SOURCE IDs and see if they appear inside any PATCH filename
    # This is slower but guarantees a match if the ID is present at all.

    for sid in source_ids:
        match_found = False
        for p_name in patch_files:
            # SIMPLEST CHECK: Is 'img_001' inside 'Luminal A_img_001_p0.png'?
            if sid in p_name:
                found_patch_origins.add(sid)
                match_found = True
                break # Move to next source ID once found

        if not match_found:
            # Optional: Print first failure to see why
            pass

    # 4. Results
    missing_slides = source_ids - found_patch_origins

    print("-" * 30)
    print(f"Summary:")
    print(f"Matches Found: {len(found_patch_origins)} / {len(source_ids)}")

    if len(missing_slides) > 0:
        print(f"Missing: {len(missing_slides)}")
        print(f"First 5 missing: {list(missing_slides)[:5]}")
    else:
        print("SUCCESS: All slides accounted for.")

# Run
debug_patches_coverage(SOURCE_DIR, PATCH_FOLDERS)

🔎 Starting DEBUG Coverage Check...
📂 Found 577 source slides.
   Examples (Source): ['img_0634', 'img_0074', 'img_0561', 'img_0250', 'img_0440']
🧩 Found 8584 patches total.
   Examples (Patches): ['Triple negative_img_0605_p0.png', 'Triple negative_img_0605_p1.png', 'Triple negative_img_0605_p2.png', 'Triple negative_img_0605_p3.png', 'Triple negative_img_0605_p4.png']

--- Testing Matches ---
------------------------------
📊 Summary:
   Matches Found: 577 / 577
✅ SUCCESS: All slides accounted for.


# __Feature extraction__


In [ ]:
!pip install transformers

In [ ]:
# Network one - for geometric information
import torch.nn as nn

class GeometricMaskEncoder(nn.Module):
    """
    The 'Intent' Pillar.
    Encodes the binary mask geometry into a compact feature vector.
    """
    def __init__(self, output_dim=128):
        super(GeometricMaskEncoder, self).__init__()
        self.features = nn.Sequential(
            # Input: 1 channel (Binary Mask)
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2), # 112x112

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), # 56x56

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)) # Forces output to 64 x 4 x 4
        )
        self.fc = nn.Linear(64 * 4 * 4, output_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

# Instantiate it once to check
mask_encoder = GeometricMaskEncoder(output_dim=128)
print("Geometric Mask Encoder initialized.")

✅ Geometric Mask Encoder initialized.


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import os
import numpy as np
from tqdm.notebook import tqdm  # Use notebook-friendly progress bar

# --- 1. DATASET CLASS ---
class PatchDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['filename']
        path = os.path.join(self.img_dir, fname)

        # Load and convert to RGB
        try:
            image = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"Warning: Could not load {path}: {e}")
            # Return a black image of correct size to avoid crashing
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image

from transformers import AutoImageProcessor, AutoModel

def get_pathology_extractor(device):
    """
    Loads the Owkin/Phikon model (ViT-Base) pre-trained on histology.
    Output dimension: 768 features (vs 512 for ResNet18).
    """
    print("Downloading/Loading Pathology Foundation Model (Phikon)...")

    # Load the processor (handles resizing/normalization automatically)
    processor = AutoImageProcessor.from_pretrained("owkin/phikon", use_fast=True)

    # Load the model
    model = AutoModel.from_pretrained("owkin/phikon")
    model.to(device)
    model.eval()

    return model, processor

#Replaced
def get_dual_models(device):
    print("Loading Phikon (Image) + Geometric (Mask) Models...")
    # Pillar 1: Image (Phikon)
    processor = AutoImageProcessor.from_pretrained("owkin/phikon", use_fast=True)
    img_model = AutoModel.from_pretrained("owkin/phikon")
    img_model.to(device).eval()

    # Pillar 2: Mask (Geometry)
    mask_model = GeometricMaskEncoder(output_dim=128)
    mask_model.to(device).eval()

    return img_model, processor, mask_model

def extract_features_dual_integration(csv_path, images_dir, masks_dir, output_dir, batch_size=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Starting Dual-Stream Extraction (Fixed) on: {device}")

    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)

    # 1. Load Models
    img_model, processor, mask_model = get_dual_models(device)

    # 2. Index Masks
    print(f"Indexing masks in {masks_dir}...")
    mask_lookup = {}
    if os.path.exists(masks_dir):
        for f in os.listdir(masks_dir):
            if "_mask_" in f:
                suffix = f.split("_mask_")[-1]
                mask_lookup[suffix] = f
    print(f"Indexed {len(mask_lookup)} masks.")

    grouped = df.groupby('original_image')

    mask_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((224, 224), antialias=True)
    ])

    with torch.no_grad():
        for slide_id, group in tqdm(grouped, desc="Processing Slides"):

            # Resume Logic
            save_name = os.path.splitext(slide_id)[0] + ".pt"
            save_path = os.path.join(output_dir, save_name)
            if os.path.exists(save_path): continue

            batch_imgs = []
            batch_masks = []
            features_list = []

            for i, (_, row) in enumerate(group.iterrows()):
                img_filename = row['filename']
                img_path = os.path.join(images_dir, img_filename)

                # Mask Lookup
                mask_filename = None
                if "_img_" in img_filename:
                    suffix = img_filename.split("_img_")[-1]
                    mask_filename = mask_lookup.get(suffix)

                if mask_filename is None:
                    mask_filename = img_filename.replace("img", "mask")

                mask_path = os.path.join(masks_dir, mask_filename)

                try:
                    # Load Image (Already Focused)
                    img_pil = Image.open(img_path).convert("RGB")

                    # Load Mask (Geometry)
                    if os.path.exists(mask_path):
                        mask_pil = Image.open(mask_path).convert("L")
                    else:
                        mask_pil = Image.new('L', img_pil.size, 0) # Fallback

                    batch_imgs.append(img_pil)
                    batch_masks.append(mask_transform(mask_pil))

                except Exception as e:
                    # --- FIX 2: Print error for the first failure to debug paths ---
                    if len(features_list) == 0 and len(batch_imgs) == 0:
                        print(f"Error loading {img_filename}: {e}")
                    pass

                # --- PROCESS BATCH ---
                is_last_item = (i == len(group) - 1)
                if len(batch_imgs) >= batch_size or (len(batch_imgs) > 0 and is_last_item):

                    inputs = processor(images=batch_imgs, return_tensors="pt").to(device)
                    feat_img = img_model(**inputs).last_hidden_state[:, 0, :]

                    t_masks = torch.stack(batch_masks).to(device)
                    feat_mask = mask_model(t_masks)

                    combined = torch.cat([feat_img, feat_mask], dim=1)
                    features_list.append(combined.cpu())

                    batch_imgs = []
                    batch_masks = []

            # Save Slide Features
            if len(features_list) > 0:
                slide_features = torch.cat(features_list, dim=0)
                label = group.iloc[0]['label']
                torch.save({
                    'features': slide_features,
                    'label': label,
                    'slide_id': slide_id
                }, save_path)
            else:
                # Debugging: Print if a slide was skipped completely
                # print(f"Skipped {slide_id}: No patches processed.")
                pass

    print(f"Completed. Features saved in: {output_dir}")

In [ ]:
import os
import pandas as pd

def rebuild_csv_from_images(image_dir, output_csv_path):
    print(f"Scanning {image_dir} to rebuild CSV...")

    if not os.path.exists(image_dir):
        print(f"Error: Directory {image_dir} does not exist!")
        return

    data = []
    files = [f for f in os.listdir(image_dir) if f.endswith(".png")]

    for filename in files:
        try:
            # 1. Split at "_img_" to separate Label from the rest
            if "_img_" in filename:
                parts = filename.split("_img_")
                label = parts[0]

                # 2. Reconstruct original image name (e.g., img_0564.png)
                # The second part is "0564_p10.png". We take the ID part.
                rest = parts[1]
                image_id = rest.split("_p")[0]
                original_image = f"img_{image_id}.png"

                data.append({
                    "filename": filename,
                    "label": label,
                    "original_image": original_image
                })
        except Exception as e:
            print(f"Skipped {filename}: {e}")

    # Create and save DataFrame
    df = pd.DataFrame(data)
    df.to_csv(output_csv_path, index=False)
    print(f"Saved {len(df)} rows to {output_csv_path}")

# 1. Rebuild Train CSV
rebuild_csv_from_images(
    image_dir="./patch/train/images",
    output_csv_path="./patch/train/train_patches.csv"
)

# 2. Rebuild Val CSV
rebuild_csv_from_images(
    image_dir="./patch/val/images",
    output_csv_path="./patch/val/val_patches.csv"
)

📂 Scanning ./patch/train/images to rebuild CSV...
✅ Saved 8102 rows to ./patch/train/train_patches.csv
📂 Scanning ./patch/val/images to rebuild CSV...
✅ Saved 482 rows to ./patch/val/val_patches.csv


In [ ]:
extract_features_dual_integration(
    csv_path="./patch/train/train_patches.csv",
    images_dir="./patch/train/images",
    masks_dir="./patch/train/masks",
    output_dir="./patch/train/features_dual",
    batch_size=64
)

🚀 Starting Dual-Stream Extraction (Fixed) on: cpu
⏳ Loading Phikon (Image) + Geometric (Mask) Models...


preprocessor_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

🔍 Indexing masks in ./patch/train/masks...
   ✅ Indexed 8102 masks.


Processing Slides:   0%|          | 0/461 [00:00<?, ?it/s]

✅ Completed. Features saved in: ./patch/train/features_dual


In [ ]:
extract_features_dual_integration(
    csv_path="./patch/val/val_patches.csv",
    images_dir="./patch/val/images",
    masks_dir="./patch/val/masks",
    output_dir="./patch/val/features_dual",
    batch_size=64
)

🚀 Starting Dual-Stream Extraction (Fixed) on: cpu
⏳ Loading Phikon (Image) + Geometric (Mask) Models...
🔍 Indexing masks in ./patch/val/masks...
   ✅ Indexed 482 masks.


Processing Slides:   0%|          | 0/116 [00:00<?, ?it/s]

✅ Completed. Features saved in: ./patch/val/features_dual


## __Alternative train/val split__
We can run the following cells if we want to have the features in a unque folders. In this way we can dynamically build (paying attention to data leakage) the train and validation sets.

In [ ]:
#Merge all the features and images in a unique folder

import os
import shutil
import pandas as pd
from tqdm.notebook import tqdm

# --- CONFIG ---
BASE_DIR = "./patch"
DIRS_TO_MERGE = ["train", "val"]
TARGET_DIR = os.path.join(BASE_DIR, "all_data")
SUBFOLDERS = ["images", "masks", "features_dual"]

# 1. Create Target Structure
for sub in SUBFOLDERS:
    os.makedirs(os.path.join(TARGET_DIR, sub), exist_ok=True)

print(f"Created target directory: {TARGET_DIR}")

# 2. Merge Files (Images, Masks, Features)
for split in DIRS_TO_MERGE:
    print(f"\n--- Merging '{split}' into 'all_data' ---")
    for sub in SUBFOLDERS:
        src_path = os.path.join(BASE_DIR, split, sub)
        dst_path = os.path.join(TARGET_DIR, sub)

        if not os.path.exists(src_path):
            continue

        files = os.listdir(src_path)
        print(f"Moving {len(files)} files from {sub}...")

        for f in tqdm(files):
            src_file = os.path.join(src_path, f)
            dst_file = os.path.join(dst_path, f)

            # Move file (Avoid overwriting if exists, though unlikely with unique names)
            if not os.path.exists(dst_file):
                shutil.move(src_file, dst_file)

# 3. Merge CSVs
print("\n--- Merging CSV Manifests ---")
csv_list = []
for split in DIRS_TO_MERGE:
    csv_path = os.path.join(BASE_DIR, split, f"{split}_patches.csv")
    if os.path.exists(csv_path):
        csv_list.append(pd.read_csv(csv_path))

if csv_list:
    full_df = pd.concat(csv_list, ignore_index=True)
    full_csv_path = os.path.join(TARGET_DIR, "all_patches.csv")
    full_df.to_csv(full_csv_path, index=False)
    print(f"Merged CSV saved to: {full_csv_path} ({len(full_df)} rows)")
else:
    print("No CSVs found to merge.")

print("\nData Consolidation Complete!")

📂 Created target directory: ./patch/all_data

--- Merging 'train' into 'all_data' ---
Moving 8102 files from images...


  0%|          | 0/8102 [00:00<?, ?it/s]

Moving 8102 files from masks...


  0%|          | 0/8102 [00:00<?, ?it/s]

Moving 461 files from features_dual...


...

Moving 482 files from masks...


  0%|          | 0/482 [00:00<?, ?it/s]

Moving 116 files from features_dual...


  0%|          | 0/116 [00:00<?, ?it/s]


--- Merging CSV Manifests ---
✅ Merged CSV saved to: ./patch/all_data/all_patches.csv (8584 rows)

✅ Data Consolidation Complete!


# __Training__


In [ ]:
class GatedAttention(nn.Module):
    def __init__(self, input_dim=256):
        super(GatedAttention, self).__init__()
        self.L = input_dim
        self.D = 64
        self.K = 1

        self.attention_V = nn.Sequential(nn.Linear(self.L, self.D), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(self.L, self.D), nn.Sigmoid())
        self.attention_weights = nn.Linear(self.D, self.K)

    def forward(self, x):
        A_V = self.attention_V(x)
        A_U = self.attention_U(x)
        A = self.attention_weights(A_V * A_U)
        A = torch.transpose(A, 1, 0)
        A = torch.softmax(A, dim=1)
        return A

class TransformerMIL(nn.Module):
    def __init__(self, num_classes=4, input_dim=896):
        super(TransformerMIL, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25)
        )

        # 2. Transformer
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=256,
                nhead=4,
                dim_feedforward=256,
                dropout=0.5,
                activation='relu'
            ),
            num_layers=2
        )

        # 3. Attention & Classifier
        self.attention = GatedAttention(input_dim=256)
        self.classifier = nn.Sequential(
            nn.Linear(256, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.fc(x).unsqueeze(1)
        x = self.transformer(x).squeeze(1)
        A = self.attention(x)
        M = torch.mm(A, x)
        return self.classifier(M), A

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
import os
import glob
from tqdm.notebook import tqdm

# --- CONFIG ---
N_FOLDS = 5
EPOCHS = 50
PATIENCE = 12
INPUT_DIM = 896
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALL_FEATURES_DIR = "./patch/all_data/features_dual"
LABEL_MAP = {'Luminal A': 0, 'Luminal B': 1, 'HER2(+)': 2, 'Triple negative': 3}

# --- 1. FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        CE_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-CE_loss)
        F_loss = ((1 - pt) ** self.gamma) * CE_loss
        if self.reduction == 'mean': return torch.mean(F_loss)
        elif self.reduction == 'sum': return torch.sum(F_loss)
        else: return F_loss

# --- DATASET ---
class InMemoryFeatureBagDataset(Dataset):
    def __init__(self, bags_list): self.bags = bags_list
    def __len__(self): return len(self.bags)
    def __getitem__(self, idx): return self.bags[idx]
def collate_features(batch): return batch[0]

# --- LOAD DATA ---
if 'all_bags' not in locals() or not all_bags:
    print("⏳ Loading features...")
    all_files = glob.glob(os.path.join(ALL_FEATURES_DIR, "*.pt"))
    all_bags = []
    for path in tqdm(all_files):
        try:
            data = torch.load(path, map_location='cpu')
            lbl = LABEL_MAP.get(data['label'], -1)
            if lbl != -1: all_bags.append((data['features'], lbl, data['slide_id']))
        except: pass

all_data_array = np.array(all_bags, dtype=object)
all_labels = np.array([b[1] for b in all_bags])

# --- TRAINING LOOP ---
print(f"Starting {N_FOLDS}-Fold Cross-Validation...")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(all_data_array, all_labels)):
    print(f"\n" + "="*40)
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print("="*40)

    # Prepare Data
    train_bags = all_data_array[train_idx].tolist()
    val_bags = all_data_array[val_idx].tolist()

    y_train = [b[1] for b in train_bags]
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights_tensor = torch.tensor(class_weights ** 0.6, dtype=torch.float).to(DEVICE)

    train_loader = DataLoader(InMemoryFeatureBagDataset(train_bags), batch_size=1, shuffle=True, collate_fn=collate_features)
    val_loader = DataLoader(InMemoryFeatureBagDataset(val_bags), batch_size=1, shuffle=False, collate_fn=collate_features)

    # Init Model & Loss
    model = TransformerMIL(num_classes=4, input_dim=INPUT_DIM).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    criterion = FocalLoss(alpha=weights_tensor, gamma=2.0)

    # Training
    best_val_f1 = 0.0
    patience_counter = 0
    best_model_state = None

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for features, label, _ in train_loader:
            features, label = features.to(DEVICE), torch.tensor([label]).to(DEVICE)
            if epoch < 15: features = features + (torch.randn_like(features) * 0.02) # TTA Noise

            optimizer.zero_grad()
            logits, _ = model(features)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validate
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for features, label, _ in val_loader:
                features = features.to(DEVICE)
                logits, _ = model(features)
                val_preds.append(torch.argmax(logits, dim=1).item())
                val_targets.append(label)

        val_f1 = f1_score(val_targets, val_preds, average='macro')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()
            patience_counter = 0
            print(f"   🌟 Epoch {epoch+1}: Best F1! {val_f1:.4f} (Loss: {train_loss/len(train_loader):.4f})")
        else:
            patience_counter += 1
            if (epoch + 1) % 5 == 0:
                print(f"      Epoch {epoch+1}: F1 {val_f1:.4f}")

        if patience_counter >= PATIENCE:
            print(f"   🛑 Early stopping at epoch {epoch+1}")
            break

    # Save Model
    save_path = f"model_5fold_focal_{fold}.pth"
    torch.save(best_model_state, save_path)
    fold_models.append(save_path)
    print(f"Fold {fold+1} Finished. Saved to {save_path}")

🎯 Target Distribution: {'Luminal A': 177, 'Luminal B': 157, 'HER2(+)': 119, 'Triple negative': 24}
🔎 Scanning Seeds: [3407, 2024, 88, 555, 123]

🌱 TESTING SEED: 3407
   ✅ Fold 1/5 done.
   ✅ Fold 2/5 done.


KeyboardInterrupt: 

# __Test data and submission generation__

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from tqdm.notebook import tqdm

# --- CONFIG ---
TEST_DIR = './test_data/'                # Where your big test images are
TEMP_PATCH_DIR = './test_patches_temp/'   # Output folder

# --- SETTINGS
PATCH_SIZE = 256
STRIDE = 256
MASK_RATIO_THRESH = 0.01

# Setup Folders
img_dest = os.path.join(TEMP_PATCH_DIR, "images")
mask_dest = os.path.join(TEMP_PATCH_DIR, "masks")
os.makedirs(img_dest, exist_ok=True)
os.makedirs(mask_dest, exist_ok=True)

# Find Images
test_files = [f for f in os.listdir(TEST_DIR) if f.startswith('img') and f.endswith(('.png', '.jpg'))]
patch_data = []

print(f"Slicing {len(test_files)} Test Slides (Natural Background)...")

for filename in tqdm(test_files):
    # 1. Match Paths
    img_path = os.path.join(TEST_DIR, filename)

    # Try generic mask replacement
    mask_name = filename.replace('img', 'mask').replace('.jpg', '.png')
    mask_path = os.path.join(TEST_DIR, mask_name)
    if not os.path.exists(mask_path):
        mask_name = filename.replace('img', 'mask') # Fallback
        mask_path = os.path.join(TEST_DIR, mask_name)

    if not os.path.exists(mask_path):
        print(f"⚠️ Mask missing for {filename}, skipping...")
        continue

    # 2. Load
    image = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image is None or mask is None: continue

    # --- NORMALIZATION ---
    # Decomment to use Macenko and Araujo
    # try:
    #    image = normalize_stain(image)
    #    image = araujo_contrast_stretching(image)
    # except:
    #    pass

    # 3. Slice Loop
    h, w, _ = image.shape
    patch_id = 0

    for y in range(0, h - PATCH_SIZE + 1, STRIDE):
        for x in range(0, w - PATCH_SIZE + 1, STRIDE):

            # Extract (Natural)
            img_patch = image[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            mask_patch = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

            # Filter Tissue
            tumor_pixels = np.count_nonzero(mask_patch)
            tumor_ratio = tumor_pixels / (PATCH_SIZE**2)

            if tumor_ratio >= MASK_RATIO_THRESH:

                # Save
                patch_name = f"{filename[:-4]}_p{patch_id}.png"

                # SAVE NATURAL IMAGE (No masking applied to pixels)
                cv2.imwrite(os.path.join(img_dest, patch_name), img_patch)
                cv2.imwrite(os.path.join(mask_dest, patch_name), mask_patch)

                # Record metadata
                patch_data.append({
                    'original_image': filename,
                    'filename': patch_name,
                    'label': 'Unknown'
                })
                patch_id += 1

# Save CSV
test_csv_path = os.path.join(TEMP_PATCH_DIR, "test_patches.csv")
test_df = pd.DataFrame(patch_data)
test_df.to_csv(test_csv_path, index=False)
print(f"Generated {len(test_df)} test patches (Natural).")

🔪 Slicing 477 Test Slides (High Fidelity 224x224)...


  0%|          | 0/477 [00:00<?, ?it/s]

✅ Generated 2099 test patches.


In [ ]:
def extract_test_features_dual(csv_path, images_dir, masks_dir, output_dir, batch_size=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Starting Test Set Extraction (Fixed) on: {device}")

    # Reset output directory
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(csv_path)

    # Load Models
    img_model, processor, mask_model = get_dual_models(device)

    # Mask Transform
    mask_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((224, 224), antialias=True)
    ])

    grouped = df.groupby('original_image')
    processed_slides = [] # Registry list

    with torch.no_grad():
        for slide_id, group in tqdm(grouped, desc="Processing Test Slides"):

            save_name = os.path.splitext(slide_id)[0] + ".pt"
            save_path = os.path.join(output_dir, save_name)

            batch_imgs = []
            batch_masks = []
            features_list = []

            for i, (_, row) in enumerate(group.iterrows()):
                filename = row['filename']

                # Test set logic: Name is identical in both folders
                img_path = os.path.join(images_dir, filename)
                mask_path = os.path.join(masks_dir, filename)

                try:
                    # 1. LOAD IMAGE
                    img_pil = Image.open(img_path).convert("RGB")

                    # 2. LOAD MASK
                    if os.path.exists(mask_path):
                        mask_pil = Image.open(mask_path).convert("L")
                    else:
                        mask_pil = Image.new('L', img_pil.size, 0) # Fallback

                    batch_imgs.append(img_pil)
                    batch_masks.append(mask_transform(mask_pil))

                except Exception as e:
                    pass

                # Check if this is the last item in the group
                is_last_item = (i == len(group) - 1)

                if len(batch_imgs) >= batch_size or (len(batch_imgs) > 0 and is_last_item):

                    # A. Image Features
                    inputs = processor(images=batch_imgs, return_tensors="pt").to(device)
                    feat_img = img_model(**inputs).last_hidden_state[:, 0, :]

                    # B. Mask Features
                    t_masks = torch.stack(batch_masks).to(device)
                    feat_mask = mask_model(t_masks)

                    # C. FUSION
                    combined = torch.cat([feat_img, feat_mask], dim=1)
                    features_list.append(combined.cpu())

                    batch_imgs = []
                    batch_masks = []

            # Save Slide Features
            if len(features_list) > 0:
                slide_features = torch.cat(features_list, dim=0)

                torch.save({
                    'features': slide_features,
                    'label': 'Unknown',
                    'slide_id': slide_id
                }, save_path)

                # Add to Registry
                processed_slides.append({
                    'slide_id': slide_id,
                    'feature_file': save_name,
                    'label': 'Unknown'
                })

    # --- SAVE CSV ---
    if len(processed_slides) > 0:
        csv_save_path = os.path.join(output_dir, "test_features.csv")
        pd.DataFrame(processed_slides).to_csv(csv_save_path, index=False)
        print(f"Generated {len(processed_slides)} feature bags.")
        print(f"Registry saved to: {csv_save_path}")
    else:
        print("No features were extracted. Check your test_patches.csv path!")

In [ ]:
import shutil

# Create output directory for test features
test_features_dir = os.path.join(TEMP_PATCH_DIR, "features")

if os.path.exists(test_features_dir):
    shutil.rmtree(test_features_dir)
os.makedirs(test_features_dir, exist_ok=True)

# Run Feature Extraction
extract_test_features_dual(
    csv_path="./test_patches_temp/test_patches.csv",
    images_dir="./test_patches_temp/images",
    masks_dir="./test_patches_temp/masks",
    output_dir="./test_patches_temp/features_dual", # Output folder
    batch_size=64
)

🚀 Starting Test Set Extraction (Fixed) on: cpu
⏳ Loading Phikon (Image) + Geometric (Mask) Models...


Processing Test Slides:   0%|          | 0/477 [00:00<?, ?it/s]

✅ Generated 477 feature bags.
📄 Registry saved to: ./test_patches_temp/features_dual/test_features.csv


## __Test for basic model__

In [ ]:
import torch
import pandas as pd
import os
from tqdm.notebook import tqdm

# --- CONFIG ---
MODEL_PATH = "" #Insert the path of the model to test
TEST_FEATURES_CSV = "./test_patches_temp/features_dual/test_features.csv"
TEST_FEATURES_DIR = "./test_patches_temp/features_dual"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_DIM = 768 + 128

# 1. Load the Best Model
print(f"Loading model from {MODEL_PATH}...")
best_model = TransformerMIL(num_classes=4, input_dim=INPUT_DIM).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
best_model.eval()

# 2. Load Test Registry
if not os.path.exists(TEST_FEATURES_CSV):
    print(f"Error: {TEST_FEATURES_CSV} not found.")
else:
    test_df = pd.read_csv(TEST_FEATURES_CSV)
    print(f"Loaded test registry with {len(test_df)} slides.")

    predictions = []
    class_names = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']

    print("Running Inference...")
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
            slide_id = row['slide_id']
            feature_path = os.path.join(TEST_FEATURES_DIR, row['feature_file'])

            try:
                # Load features
                data = torch.load(feature_path, map_location=DEVICE)
                features = data['features'].to(DEVICE)

                # Forward Pass
                logits, _ = best_model(features)

                # Prediction
                pred_idx = torch.argmax(logits, dim=1).item()

                predictions.append({
                    'slide_id': slide_id,
                    'label_str': class_names[pred_idx]
                })
            except Exception as e:
                print(f"Error predicting {slide_id}: {e}")

    # 3. Format Submission (Textual Labels + String IDs)
    submission_df = pd.DataFrame(predictions)

    # A. Use Textual Labels directly
    submission_df['label'] = submission_df['label_str']

    # B. Use String IDs directly (e.g., "img_0012.png")
    submission_df['sample_index'] = submission_df['slide_id']

    # C. Final Cleanup
    submission_df = submission_df[['sample_index', 'label']]
    submission_df = submission_df.sort_values('sample_index')

    # Save
    save_name = "submission_textual.csv"
    submission_df.to_csv(save_name, index=False)

    print(f"\nSUCCESS! Submission saved to '{save_name}'")
    print(f"   Shape: {submission_df.shape}")
    print(submission_df.head())

📥 Loading model from best_mil_model_f1_seed490.pth...
📂 Loaded test registry with 477 slides.
🚀 Running Inference...


  0%|          | 0/477 [00:00<?, ?it/s]


✅ SUCCESS! Submission saved to 'submission_textual.csv'
   Shape: (477, 2)
   sample_index            label
0  img_0000.png        Luminal A
1  img_0001.png        Luminal B
2  img_0002.png        Luminal A
3  img_0003.png  Triple negative
4  img_0004.png        Luminal B


## __Test using cross validation training__

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
import re
from tqdm.notebook import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
N_FOLDS = 5
INPUT_DIM = 896
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
TEST_FEATURES_CSV = "./test_patches_temp/features_dual/test_features.csv"
TEST_DIR = "./test_patches_temp/features_dual"

# Model prefix
# Remember to rename it if we apply some changing in the procedure of cross-validation
MODEL_PREFIX = "model_5fold_focal_"

# Classes
CLASS_NAMES = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']

# ==========================================
# 1. LOAD MODELS
# ==========================================
print(f"Loading {N_FOLDS} K-Fold Models...")
models = []

for fold in range(N_FOLDS):
    path = f"{MODEL_PREFIX}_{fold}.pth"
    if not os.path.exists(path):
        print(f"Warning: Model file {path} not found. Skipping.")
        continue

    # Initialize model architecture
    # Ensure TransformerMIL class is defined in your notebook before running this
    model = TransformerMIL(num_classes=4, input_dim=INPUT_DIM).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    models.append(model)

if not models:
    raise RuntimeError("No models were loaded! Check your file paths.")

# ==========================================
# 2. INFERENCE LOOP (SOFT VOTING)
# ==========================================
print(f"Running Ensemble Inference on Test Set...")

if not os.path.exists(TEST_FEATURES_CSV):
    raise FileNotFoundError("Test features CSV not found.")

df_test = pd.read_csv(TEST_FEATURES_CSV)
predictions = []

with torch.no_grad():
    for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
        feature_path = os.path.join(TEST_DIR, row['feature_file'])
        try:
            # Load features
            data = torch.load(feature_path, map_location=DEVICE)
            features = data['features'].to(DEVICE)

            # Collect probabilities from all models
            probs_sum = torch.zeros(1, 4).to(DEVICE)

            for m in models:
                logits, _ = m(features)
                probs_sum += F.softmax(logits, dim=1)

            # Average probabilities (Soft Voting)
            avg_probs = probs_sum / len(models)
            pred_idx = torch.argmax(avg_probs).item()

            predictions.append({
                'sample_index': row['slide_id'],
                'label': CLASS_NAMES[pred_idx]
            })
        except Exception as e:
            print(f"Error processing {row['slide_id']}: {e}")
            # Fallback to majority class if file read fails
            predictions.append({'sample_index': row['slide_id'], 'label': 'Luminal A'})

# ==========================================
# 3. SAVE SUBMISSION
# ==========================================
sub_df = pd.DataFrame(predictions)

# Custom sorting to ensure numerical order (e.g., slide_1, slide_2, slide_10)
sub_df['sort_key'] = sub_df['sample_index'].apply(
    lambda x: int(re.search(r'\d+', str(x)).group()) if re.search(r'\d+', str(x)) else 0
)
sub_df = sub_df.sort_values('sort_key').drop(columns=['sort_key']).reset_index(drop=True)

# Save to CSV
save_name = "submission_kfold_final.csv"
sub_df.to_csv(save_name, index=False)

print(f"\nSubmission Saved: {save_name}")
print("\n📊 Predicted Class Distribution:")
print(sub_df['label'].value_counts())